In [1]:
import os
import time
import glob
import pandas as pd
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from tqdm.notebook import tqdm # Added for progress bars in Jupyter

# 1. Setup and Initialization
load_dotenv(dotenv_path='../.env', override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
encoding = tiktoken.get_encoding("cl100k_base")

MODEL = "text-embedding-3-small"
MAX_TOKENS = 8191
BATCH_SIZE = 1000         # Maximum safe payload size
SAVE_INTERVAL = 50000     # Save a pickle file every 50k rows
CHECKPOINT_DIR = "../data/checkpoints3"
TPM_LIMIT = 900000  # 900k to stay safely under the 1M limit

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Setup complete. Checkpoint directory ready.")


Setup complete. Checkpoint directory ready.


In [2]:
# 2. Load your actual data here
# df = pd.read_csv("your_massive_dataset.csv")

# Simulated dataframe for demonstration
df = pd.read_csv("../data/unified_threads.csv")

def prepare_text(text):
    text = str(text).replace("\n", " ")
    tokens = encoding.encode(text, disallowed_special=())
    if len(tokens) > MAX_TOKENS:
        return encoding.decode(tokens[:MAX_TOKENS])
    return text

print("Step 1: Formatting texts to enforce token limits...")

# Using tqdm to show a progress bar for the apply function
tqdm.pandas(desc="Preparing text")
df['safe_content'] = df['content'].progress_apply(prepare_text)
texts_to_embed = df['safe_content'].tolist()

print(f"Total texts prepared for embedding: {len(texts_to_embed)}")
print(df.head)

Step 1: Formatting texts to enforce token limits...


Preparing text:   0%|          | 0/8120 [00:00<?, ?it/s]

Total texts prepared for embedding: 8120
<bound method NDFrame.head of      interaction_type   subreddit       id clean_parent_id clean_root_id  \
0                post  philosophy   100cxy             NaN        100cxy   
1                post  philosophy  100zfxn             NaN       100zfxn   
2             comment  philosophy  j2kvtfk         100zfxn       100zfxn   
3             comment  philosophy  j2kw3ny         100zfxn       100zfxn   
4             comment  philosophy  j2kwgwj         100zfxn       100zfxn   
...               ...         ...      ...             ...           ...   
8115          comment  philosophy  j2qark7         j2dr52m        zvnq0i   
8116          comment  philosophy  j2qbe7t         j1rizp0        zvnq0i   
8117             post  philosophy    zw95m             NaN         zw95m   
8118             post  philosophy    zwwcj             NaN         zwwcj   
8119             post  philosophy    zylw7             NaN         zylw7   

                

In [3]:
print(f"Starting API processing of {len(texts_to_embed)} rows...")
current_chunk_embeddings = []
chunk_index = 0

tokens_in_window = 0
window_start_time = time.time()

# 1. Dynamically group texts into safe request sizes
MAX_TOKENS_PER_REQUEST = 250000 # Keep a safe buffer below the 300,000 limit
MAX_ROWS_PER_REQUEST = 1000 # Hard cap the array size per OpenAI limits

batches = []
current_batch = []
current_batch_tokens = 0

for text in texts_to_embed:
    # Get exact token count for this specific text
    token_count = len(encoding.encode(text, disallowed_special=()))
    
    # If adding the next text pushes us over the token limit OR row limit, seal the batch
    if current_batch_tokens + token_count > MAX_TOKENS_PER_REQUEST or len(current_batch) >= MAX_ROWS_PER_REQUEST:
        batches.append((current_batch, current_batch_tokens))
        current_batch = []
        current_batch_tokens = 0
        
    current_batch.append(text)
    current_batch_tokens += token_count

# Don't forget the last partial batch
if current_batch:
    batches.append((current_batch, current_batch_tokens))

print(f"Divided data into {len(batches)} dynamic batches.")

# 2. Process the dynamic batches
processed_count = 0
for batch, batch_tokens in batches:
    
    # Check if this batch will breach our 900k TPM safety limit
    if tokens_in_window + batch_tokens > TPM_LIMIT:
        elapsed_time = time.time() - window_start_time
        
        # If a full minute hasn't passed yet, sleep for the remaining time
        if elapsed_time < 60:
            sleep_time = 60 - elapsed_time
            print(f"  [Rate Limit Paused] Approaching 1M limit. Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
        
        # Reset the token counter and start a new 60-second window
        tokens_in_window = 0
        window_start_time = time.time()

    # Make the API request 
    response = client.embeddings.create(
        input=batch,
        model=MODEL
    )
    
    # Log the tokens and add to our current window count
    tokens_in_window += batch_tokens
    
    batch_embeddings = [item.embedding for item in response.data]
    current_chunk_embeddings.extend(batch_embeddings)
    processed_count += len(batch)
    
    # Trigger a save when we hit the interval, or reach the last batch
    if len(current_chunk_embeddings) >= SAVE_INTERVAL or processed_count == len(texts_to_embed):
        start_idx = chunk_index * SAVE_INTERVAL
        end_idx = start_idx + len(current_chunk_embeddings)
        
        df_chunk = df.iloc[start_idx:end_idx].copy()
        df_chunk['embedding'] = current_chunk_embeddings
        
        filename = f"{CHECKPOINT_DIR}/embeddings_part_{chunk_index}.pkl"
        df_chunk.to_pickle(filename)
        print(f"Saved checkpoint securely to: {filename} (Rows {start_idx} to {end_idx})")
        
        current_chunk_embeddings = []
        chunk_index += 1

# 3. Combine Checkpoints
print("\nMerging all checkpoints from local directory...")
all_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/*.pkl"))

if not all_files:
    print("No checkpoint files found to merge.")
else:
    df_final = pd.concat([pd.read_pickle(f) for f in all_files], ignore_index=True)

    if 'safe_content' in df_final.columns:
        df_final = df_final.drop(columns=['safe_content'])
    final_path = "../data/final_reddit_openai_embeddings2.pkl"
    df_final.to_pickle(final_path)

    print(f"\nSuccess! Final dataset saved to: {final_path}")

Starting API processing of 8120 rows...
Divided data into 11 dynamic batches.
  [Rate Limit Paused] Approaching 1M limit. Sleeping for 42.30 seconds...
  [Rate Limit Paused] Approaching 1M limit. Sleeping for 52.05 seconds...
  [Rate Limit Paused] Approaching 1M limit. Sleeping for 43.77 seconds...
Saved checkpoint securely to: ../data/checkpoints3/embeddings_part_0.pkl (Rows 0 to 8120)

Merging all checkpoints from local directory...

Success! Final dataset saved to: ../data/final_reddit_openai_embeddings2.pkl


In [4]:
print("\nStep 3: Combining checkpoints into final pickle file...")

# 4. Merge all checkpoint files
all_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/*.pkl"))

if not all_files:
    print("No checkpoint files found to merge.")
else:
    print(f"Found {len(all_files)} checkpoint files. Merging...")
    
    # Load and concatenate all pickle files
    df_final = pd.concat([pd.read_pickle(f) for f in tqdm(all_files, desc="Merging files")], ignore_index=True)
    
    # Drop the temporary column and save the master file
    if 'safe_content' in df_final.columns:
        df_final = df_final.drop(columns=['safe_content'])
        
    final_filename = "../data/final_reddit_embeddings2.pkl"
    df_final.to_pickle(final_filename)
    
    print(f"Success! Final dataset saved as '{final_filename}' with {len(df_final)} total rows.")



Step 3: Combining checkpoints into final pickle file...
Found 1 checkpoint files. Merging...


Merging files:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Final dataset saved as '../data/final_reddit_embeddings2.pkl' with 8120 total rows.


In [5]:
df_final

,interaction_type,subreddit,id,clean_parent_id,clean_root_id,author,created_utc_dt,content,score,label,embedding
0,post,philosophy,100cxy,NaN,100cxy,thesacred,2012-09-17 05:19:46,"Can someone explain to me the ""hard problem of...",82,1,"[0.0374755859375, 0.0074920654296875, -0.01593..."
1,post,philosophy,100zfxn,NaN,100zfxn,_Zirath_,2023-01-02 01:24:14,Atheistic Naturalism does not offer any long-t...,0,1,"[0.0029163360595703125, 0.04571533203125, 0.00..."
2,comment,philosophy,j2kvtfk,100zfxn,100zfxn,Ill_Sound621,2023-01-02 02:07:45,This sounds like Pascal wager with extra steps...,27,1,"[-0.00860595703125, 0.0007758140563964844, -0...."
3,comment,philosophy,j2kw3ny,100zfxn,100zfxn,catnapspirit,2023-01-02 02:09:54,"Yeah, this is still just Pascal's Wager.\n\nAl...",45,1,"[0.0269775390625, 0.01216888427734375, 0.04650..."
4,comment,philosophy,j2kwgwj,100zfxn,100zfxn,coyote-1,2023-01-02 02:12:43,An entire region suddenly floods due to stagge...,19,1,"[0.003940582275390625, 0.048065185546875, 0.00..."
...,...,...,...,...,...,...,...,...,...,...,...
8115,comment,philosophy,j2qark7,j2dr52m,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:42:48,My individuality is within my genes. My code i...,1,1,"[0.04779052734375, 0.0126495361328125, 0.00535..."
8116,comment,philosophy,j2qbe7t,j1rizp0,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:48:10,Those who live in the urban areas with the lig...,2,1,"[0.018341064453125, -0.0196685791015625, 0.001..."
8117,post,philosophy,zw95m,NaN,zw95m,fudgecrackers,2012-09-14 20:58:54,What if we refused to see others as inferior; ...,109,1,"[0.036590576171875, -0.03399658203125, 0.02586..."
8118,post,philosophy,zwwcj,NaN,zwwcj,gnomicarchitecture,2012-09-15 04:25:12,Greatest philosophers of all time. \n\nI'm in ...,7,1,"[0.0015459060668945312, -0.005352020263671875,..."


In [6]:
DATA_DIR = "../data"

In [7]:
df

,interaction_type,subreddit,id,clean_parent_id,clean_root_id,author,created_utc_dt,content,score,label,safe_content
0,post,philosophy,100cxy,NaN,100cxy,thesacred,2012-09-17 05:19:46,"Can someone explain to me the ""hard problem of...",82,1,"Can someone explain to me the ""hard problem of..."
1,post,philosophy,100zfxn,NaN,100zfxn,_Zirath_,2023-01-02 01:24:14,Atheistic Naturalism does not offer any long-t...,0,1,Atheistic Naturalism does not offer any long-t...
2,comment,philosophy,j2kvtfk,100zfxn,100zfxn,Ill_Sound621,2023-01-02 02:07:45,This sounds like Pascal wager with extra steps...,27,1,This sounds like Pascal wager with extra steps...
3,comment,philosophy,j2kw3ny,100zfxn,100zfxn,catnapspirit,2023-01-02 02:09:54,"Yeah, this is still just Pascal's Wager.\n\nAl...",45,1,"Yeah, this is still just Pascal's Wager. Also..."
4,comment,philosophy,j2kwgwj,100zfxn,100zfxn,coyote-1,2023-01-02 02:12:43,An entire region suddenly floods due to stagge...,19,1,An entire region suddenly floods due to stagge...
...,...,...,...,...,...,...,...,...,...,...,...
8115,comment,philosophy,j2qark7,j2dr52m,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:42:48,My individuality is within my genes. My code i...,1,1,My individuality is within my genes. My code i...
8116,comment,philosophy,j2qbe7t,j1rizp0,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:48:10,Those who live in the urban areas with the lig...,2,1,Those who live in the urban areas with the lig...
8117,post,philosophy,zw95m,NaN,zw95m,fudgecrackers,2012-09-14 20:58:54,What if we refused to see others as inferior; ...,109,1,What if we refused to see others as inferior; ...
8118,post,philosophy,zwwcj,NaN,zwwcj,gnomicarchitecture,2012-09-15 04:25:12,Greatest philosophers of all time. \n\nI'm in ...,7,1,Greatest philosophers of all time. I'm in a ...
